# Track 1 · 08 THINGS-EEG2 prepare — warm-up data onto Drive, small

**One job.** Put the warm-up corpus on Drive in the form `track1_07_eegnet` trains on:
preprocessed windows plus the DINOv2 target table. Nothing large touches Drive.

Self-contained: upload this one file.

---

## Why this dataset, and why this way

The warm-up leaderboard scores on THINGS-EEG2 (Gifford 2022): 63-channel research-grade
EEG, 10 people, 1000 Hz. Our Alljoined-trained EEGNet transferred to it almost intact
(4.02% warm-up top-5 against 4.33% locally), so training on it directly should lift
the warm-up score.

The EEG comes as **one zip per person, 9–13 GB each** (117 GB in total). Each zip holds a
separate file per session, and a zip's index sits at its end, so single files can be
pulled out with HTTP range requests without downloading the rest. This notebook takes
**one training session from each of the 10 people**, processes it on Colab's temporary
disk, and keeps only a sample of finished windows.

A bit from everyone rather than all of one person: the warm-up scores all 10 people, and
more people generalises better, which is what the sealed phase needs.

## Preprocessing is the harness's, exactly

The scoring harness prepares data with neuralbench's default pipeline, not with any model's
overrides. Matching it is the whole point, so it is replicated in neuralset's order:

| step | value |
|---|---|
| notch | 50 and 60 Hz plus harmonics below min(Nyquist, 301) |
| band-pass | 0.1 – 75 Hz at the native 1000 Hz |
| resample | 120 Hz |
| scale | sklearn `RobustScaler` per channel over the whole recording |
| window | −0.2 → 0.8 s, 120 samples |
| baseline | subtract the mean of the pre-stimulus 0.2 s |
| clamp | ±20 |

Trials are read from the trigger channel the way neuralfetch does (`mne.find_events`).
Trigger codes are 1-based indices into `image_metadata.npy`; code `99999` marks
attention-check trials, which are not images and are dropped.

## Licences

EEG: CC-BY-4.0. **Images: research use only, no commercial use, no redistribution**
(THINGS database licence). They are downloaded to Colab's temporary disk, embedded, and
deleted. Only the 1536-d vectors reach Drive; the images never go to Drive, the repo or a
submission zip.

## Cost

About 20 GB downloaded to temporary disk, roughly 0.5 GB written to Drive, and on the
order of 45 minutes. Resumable: a person already on Drive is skipped.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

SUBJECTS = list(range(1, 11))     # take one training session from each
TRAIN_SESSION = 1
TRAIN_WINDOWS_PER_SUBJECT = 6000  # sampled from ~16,500 per session; 6,000 -> ~90 MB each.
                                  # Each size gets its own folder (2,000 -> windows/,
                                  # others -> windows_<n>/), so earlier sets are kept.

TEST_SUBJECTS = [1, 2, 3]         # one test session each, for a local score that uses the
TEST_SESSION = 1                  # same 200 images the warm-up is scored on
TEST_WINDOWS_PER_SUBJECT = 2000

IMAGE_SETS = ['train', 'test']    # which image sets to embed

# Target space: fixed by the track config (neuralbench/tasks/eeg/image/config.yaml).
MODEL_NAME = 'facebook/dinov2-giant'
DEPTH, IMSIZE = 0.6667, 518
EMBED_BATCH = 32

# Harness preprocessing (neuralbench defaults + image task).
SFREQ = 120.0
NOTCH = (50.0, 60.0)
BANDPASS = (0.1, 75.0)
TMIN, DURATION = -0.2, 1.0
BASELINE = (-0.2, 0.0)
CLAMP = 20.0

WORK = '/content/things_work'     # temporary disk; wiped with the session
SEED = 33

In [ ]:
%pip install -q mne transformers torchvision pillow

import importlib
for mod in ('mne', 'transformers', 'torchvision', 'torch'):
    print(f'{mod:12} {getattr(importlib.import_module(mod), "__version__", "ok")}')

In [ ]:
import gc, io, json, os, shutil, time, urllib.request, warnings, zipfile
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, mne, torch
mne.set_log_level('ERROR')

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
OUT = ROOT / 'Track1' / 'things_eeg2'
WIN = OUT / 'windows'                  # test windows, and the original 2,000 train set
WIN.mkdir(parents=True, exist_ok=True)
WIN_TRAIN = WIN if TRAIN_WINDOWS_PER_SUBJECT == 2000 else OUT / f'windows_{TRAIN_WINDOWS_PER_SUBJECT}'
WIN_TRAIN.mkdir(parents=True, exist_ok=True)
WORK = Path(WORK)
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
rng = np.random.default_rng(SEED)

FIGSHARE = 'https://api.figshare.com/v2/articles/18470912/files?page_size=200'
OSF_IMAGES = 'https://api.osf.io/v2/nodes/y63gw/files/osfstorage/'
print(f'Drive output -> {OUT}\ntrain windows -> {WIN_TRAIN}\ntemporary     -> {WORK}\ndevice {DEVICE}')

## Step 1 — pull single session files out of the remote zips

Each chunk request goes through Figshare's download link, which redirects to a freshly
signed storage URL. Caching the signed URL does not work: it expires after a few chunks.

In [ ]:
class HTTPRange(io.RawIOBase):
    """Seekable read-only file over HTTP range requests, with retries."""

    def __init__(self, url, size):
        self.url, self.size, self.pos = url, size, 0

    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos

    def seek(self, off, whence=0):
        self.pos = {0: off, 1: self.pos + off, 2: self.size + off}[whence]
        return self.pos

    def readinto(self, b):
        if self.pos >= self.size:
            return 0
        end = min(self.pos + len(b), self.size) - 1
        for attempt in range(6):
            try:
                req = urllib.request.Request(self.url, headers={'Range': f'bytes={self.pos}-{end}'})
                data = urllib.request.urlopen(req, timeout=180).read()
                break
            except Exception:
                if attempt == 5:
                    raise
                time.sleep(2 ** attempt)
        b[:len(data)] = data
        self.pos += len(data)
        return len(data)


ZIPS = {f['name']: f for f in json.load(urllib.request.urlopen(FIGSHARE, timeout=60))}


def fetch_session(subject, session, split):
    """Extract one session file to temporary disk; return its path."""
    name = 'raw_eeg_training.npy' if split == 'train' else 'raw_eeg_test.npy'
    member = f'sub-{subject:02d}/ses-{session:02d}/{name}'
    dst = WORK / member
    if dst.exists():
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    zf = ZIPS[f'sub-{subject:02d}.zip']
    z = zipfile.ZipFile(io.BufferedReader(HTTPRange(zf['download_url'], zf['size']),
                                          buffer_size=32 << 20))
    t0 = time.time()
    with z.open(member) as src, open(dst.with_suffix('.part'), 'wb') as out:
        shutil.copyfileobj(src, out, 32 << 20)
    dst.with_suffix('.part').rename(dst)
    print(f'    fetched {member} ({dst.stat().st_size / 1e9:.2f} GB) in {time.time() - t0:.0f}s')
    return dst


def osf_file(name):
    items = json.load(urllib.request.urlopen(OSF_IMAGES, timeout=60))['data']
    url = next(i['links']['download'] for i in items if i['attributes']['name'] == name)
    dst = WORK / name
    if not dst.exists():
        urllib.request.urlretrieve(url, dst)
    return dst


meta = np.load(osf_file('image_metadata.npy'), allow_pickle=True).item()
IMAGE_KEYS = {s: [f'{c}/{f}' for c, f in zip(meta[f'{s}_img_concepts'], meta[f'{s}_img_files'])]
              for s in ('train', 'test')}
print(f'{len(ZIPS)} subject zips; {len(IMAGE_KEYS["train"])} training and '
      f'{len(IMAGE_KEYS["test"])} test images in the metadata')

## Step 2 — preprocess exactly as the harness does, keep a sample of windows

In [ ]:
N_TIMES = int(round(DURATION * SFREQ))
PRE = int(round(-TMIN * SFREQ))
B0, B1 = (int(round((b - TMIN) * SFREQ)) for b in BASELINE)


def harness_windows(npy_path, split):
    d = np.load(npy_path, allow_pickle=True).item()
    names = list(d['ch_names'])
    info = mne.create_info(names, sfreq=d['sfreq'], ch_types=d['ch_types'])
    raw = mne.io.RawArray(d['raw_eeg_data'], info, verbose=False)
    del d
    events = mne.find_events(raw, stim_channel='stim', verbose=False)   # as neuralfetch does
    raw.pick('eeg')
    nyq = raw.info['sfreq'] / 2
    freqs = sorted({f for base in NOTCH for f in np.arange(base, min(nyq, 301), base)})
    raw.notch_filter(freqs, phase='zero', filter_length='auto', verbose=False)
    lo, hi = BANDPASS
    raw.filter(lo, hi if hi < nyq else None, verbose=False)
    onsets = events[:, 0] / raw.info['sfreq']
    codes = events[:, 2]
    raw.resample(SFREQ, verbose=False)
    x = raw.get_data()
    ch = raw.ch_names
    del raw
    med = np.median(x, axis=1, keepdims=True)
    iqr = np.subtract(*np.percentile(x, [75, 25], axis=1))[:, None]
    iqr[iqr == 0] = 1.0
    x = ((x - med) / iqr).astype('float32')

    n_img = len(IMAGE_KEYS[split])
    keep = (codes >= 1) & (codes <= n_img)                  # drops 99999 attention checks
    s = np.round(onsets[keep] * SFREQ).astype(int) - PRE
    keys = np.array([IMAGE_KEYS[split][c - 1] for c in codes[keep]])
    ok = (s >= 0) & (s + N_TIMES <= x.shape[1])
    s, keys = s[ok], keys[ok]
    X = np.stack([x[:, i:i + N_TIMES] for i in s])
    X = X - X[:, :, B0:B1].mean(axis=2, keepdims=True)
    return np.clip(X, -CLAMP, CLAMP), keys, ch, int((~keep).sum())


def prepare(subject, session, split, n_keep):
    out = (WIN_TRAIN if split == 'train' else WIN) / f'sub-{subject:02d}_ses-{session:02d}_{split}.npz'
    if out.exists():
        print(f'  sub-{subject:02d} {split}: already on Drive, skipped')
        return
    t0 = time.time()
    path = fetch_session(subject, session, split)
    X, keys, ch, dropped = harness_windows(path, split)
    idx = np.sort(rng.choice(len(X), min(n_keep, len(X)), replace=False))
    np.savez(out, X=X[idx].astype('float16'), keys=keys[idx],
             subject=subject, session=session, split=split, channels=np.array(ch))
    path.unlink()                                          # free the temporary disk
    print(f'  sub-{subject:02d} {split}: {len(X):,} windows -> kept {len(idx):,}, '
          f'dropped {dropped} attention checks, {out.stat().st_size / 1e6:.0f} MB, '
          f'{time.time() - t0:.0f}s')
    del X, keys
    gc.collect()


print('training sessions')
for sub in SUBJECTS:
    prepare(sub, TRAIN_SESSION, 'train', TRAIN_WINDOWS_PER_SUBJECT)
print('test sessions')
for sub in TEST_SUBJECTS:
    prepare(sub, TEST_SESSION, 'test', TEST_WINDOWS_PER_SUBJECT)

## Step 3 — the target table

The same DINOv2-giant specification as `track1_01_prepare`, verified there against
neuralset's own extractor (max |diff| 0.0): hidden state 26 of 41, mean over tokens,
resize the shorter side to 518 before the HF processor. Images are read from temporary
disk and deleted afterwards. Progress is kept in shards so a dropped session resumes.

In [ ]:
import transformers
from PIL import Image as PILImage
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModel

_tf = transforms.Compose([transforms.Resize(IMSIZE), transforms.ToTensor()])
_proc = AutoImageProcessor.from_pretrained(MODEL_NAME, do_rescale=False)
_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()


@torch.no_grad()
def embed(images):
    inputs = _proc(images=[_tf(im).float() for im in images], return_tensors='pt')
    pv = inputs['pixel_values']
    if pv.isnan().any():
        pv[pv.isnan()] = 0
        inputs['pixel_values'] = pv.float()
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    states = _model(**inputs, output_hidden_states=True).hidden_states
    stacked = torch.cat([s.unsqueeze(1) for s in states], dim=1)
    idx = int(DEPTH * (stacked.shape[1] - 1))
    return stacked[:, idx].mean(dim=1).float().cpu().numpy(), idx, stacked.shape[1]


parts = OUT / 'embedding_parts'
parts.mkdir(exist_ok=True)
done = set()
for f in parts.glob('part_*.npz'):
    done |= set(np.load(f)['keys'].tolist())
if (OUT / 'image_embeddings.npz').exists():
    # A finished table replaces its shards, so without this a rerun would download and
    # embed all 16,740 images again.
    done |= set(np.load(OUT / 'image_embeddings.npz')['keys'].tolist())

for split in IMAGE_SETS:
    folder = 'training_images' if split == 'train' else 'test_images'
    todo = [k for k in IMAGE_KEYS[split] if k not in done]
    print(f'{split}: {len(IMAGE_KEYS[split]) - len(todo):,} already embedded, {len(todo):,} to go')
    if not todo:
        continue
    zpath = osf_file(f'{folder}.zip')
    with zipfile.ZipFile(zpath) as z:
        z.extractall(WORK)
    zpath.unlink()
    t0, n_part = time.time(), len(list(parts.glob('part_*.npz')))
    for start in range(0, len(todo), 2000):
        chunk = todo[start:start + 2000]
        vecs = []
        for i in range(0, len(chunk), EMBED_BATCH):
            ims = [PILImage.open(WORK / folder / k).convert('RGB')
                   for k in chunk[i:i + EMBED_BATCH]]
            v, layer, n_layers = embed(ims)
            vecs.append(v)
        np.savez(parts / f'part_{n_part:04d}.npz', keys=np.array(chunk),
                 emb=np.concatenate(vecs).astype('float32'))
        n_part += 1
        n = start + len(chunk)
        print(f'  {n:,}/{len(todo):,}  {n / (time.time() - t0):.1f} img/s  '
              f'layer {layer} of {n_layers}')
    shutil.rmtree(WORK / folder)                           # images leave the machine's disk

## Step 4 — consolidate and check

In [ ]:
keys, mats = [], []
if (OUT / 'image_embeddings.npz').exists():
    z = np.load(OUT / 'image_embeddings.npz')
    keys += z['keys'].tolist(); mats.append(z['emb'])
for f in sorted(parts.glob('part_*.npz')):
    z = np.load(f)
    keys += z['keys'].tolist(); mats.append(z['emb'])
order = {k: i for i, k in enumerate(keys)}
uniq = sorted(order)
emb = np.concatenate(mats)[[order[k] for k in uniq]]
table = OUT / 'image_embeddings.npz'
np.savez(table, keys=np.array(uniq), emb=emb, model=MODEL_NAME, depth=DEPTH, imsize=IMSIZE)

files = sorted(WIN_TRAIN.glob('*_train.npz')) + sorted(WIN.glob('*_test.npz'))
used = set()
counts = {'train': 0, 'test': 0}
channels = None
for f in files:
    z = np.load(f, allow_pickle=True)
    used |= set(z['keys'].tolist())
    counts[str(z['split'])] += len(z['keys'])
    channels = channels or list(z['channels'])
missing = sorted(used - set(uniq))
norms = np.linalg.norm(emb, axis=1)
test_imgs = {k for f in files if '_test' in f.name for k in np.load(f)['keys'].tolist()}

checks = [
    ('table', emb.shape[1] == 1536 or not MODEL_NAME.endswith('giant'),
     f'{emb.shape} from {MODEL_NAME}'),
    ('every window has a target', not missing, f'{len(missing)} missing'),
    ('finite, no zero rows', bool(np.isfinite(emb).all() and (norms > 0).all()),
     f'norms {norms.min():.2f}..{norms.max():.2f}'),
    ('windows', counts['train'] > 0,
     f"{counts['train']:,} train / {counts['test']:,} test over {len(files)} files"),
    ('channels', channels is not None and len(channels) == 63, f'{len(channels or [])}'),
    ('test gallery', len(test_imgs) in (0, 200), f'{len(test_imgs)} images'),
]
for name, ok, detail in checks:
    print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")

if all(c[1] for c in checks):
    shutil.rmtree(parts, ignore_errors=True)               # the table replaces the shards
(OUT / 'manifest.json').write_text(json.dumps(dict(
    dataset='THINGS-EEG2 (gifford2022large)', licence_eeg='CC-BY-4.0',
    licence_images='research only, no redistribution; images not stored',
    subjects=SUBJECTS, train_session=TRAIN_SESSION, test_subjects=TEST_SUBJECTS,
    counts=counts, channels=channels, sfreq=SFREQ, n_times=N_TIMES,
    preprocessing=dict(notch=NOTCH, bandpass=BANDPASS, scaler='RobustScaler',
                       baseline=BASELINE, clamp=CLAMP, window=(TMIN, TMIN + DURATION)),
    target=dict(model=MODEL_NAME, depth=DEPTH, imsize=IMSIZE, token_aggregation='mean')),
    indent=2))
used_mb = sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) / 1e6
print(f'\nDrive used by things_eeg2: {used_mb:.0f} MB')
print(f'Next: track1_07_eegnet with DATA_SOURCE = "things" and '
      f'THINGS_TRAIN_WINDOWS = {TRAIN_WINDOWS_PER_SUBJECT}.')